In [ ]:
# =============================================================================
# CELL 1 — BULLETPROOF SYSTEM SETUP & SELF-RESTARTING RESOLVER
# =============================================================================
# Run ONCE. Installs packages and auto-restarts the kernel.
# After restart: skip Cell 1, go straight to Cell 2!
# =============================================================================
import subprocess, sys, os

CLONE_DIR    = '/kaggle/working/autopilot'
PIPELINE_DIR = '/kaggle/working/autopilot/autopilot_pipeline'

# -- 0. Pin numpy FIRST to 1.26.4 ──────────────────────────────────────────
# Kaggle's pre-installed PyTorch is compiled against numpy 1.x.
# chatterbox-tts and transformers>=4.42 pull in numpy 2.x which causes:
#   ValueError: numpy.dtype size changed, may indicate binary incompatibility
# We must pin numpy BEFORE those installs happen.
print('[0/6] Pinning numpy==1.26.4 (prevents numpy 2.x binary mismatch)...')
r_np = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'numpy==1.26.4'],
    capture_output=True, text=True
)
print('  ' + ('OK  numpy pinned to 1.26.4' if r_np.returncode == 0 else 'WARN: ' + r_np.stderr[-200:]))



# Pass this to ALL subprocess calls so chatterbox/transformers use eager attention
_SETUP_ENV = {**os.environ, 'TRANSFORMERS_ATTN_IMPLEMENTATION': 'eager'}
# Correct Wan2.1 model ID (Wan-AI org, not old Wan-Video)
os.environ['WAN21_MODEL_ID'] = 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers'
_SETUP_ENV['WAN21_MODEL_ID'] = 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers'

# ── Fast health-check: if env is already good, skip the whole setup ──────────
env_ok = False
try:
    import numpy as np
    import scipy.special
    import diffusers
    from diffusers import WanPipeline
    scipy.special.sph_legendre_p(0, 0, 0)
    if np.__version__ == "1.26.4" and hasattr(diffusers, "WanPipeline"):
        env_ok = True
except Exception:
    pass

if env_ok:
    print("=" * 60)
    print("✅ Environment already verified — all checks passed!")
    print("🚀 Proceed directly to Cell 2.")
    print("=" * 60)
else:
    print("🔧 Environment needs setup. Running installer...")

    # ── 1. Clone / Pull 'main' branch ──────────────────────────────────
    print("\n[1/3] Getting latest code from branch 'main'...")
    if os.path.exists(CLONE_DIR):
        subprocess.run(['git', '-C', CLONE_DIR, 'fetch', 'origin', 'main:main'],
                       capture_output=True, env=_SETUP_ENV)
        subprocess.run(['git', '-C', CLONE_DIR, 'checkout', 'main'],
                       capture_output=True, env=_SETUP_ENV)
        r = subprocess.run(['git', '-C', CLONE_DIR, 'pull', '--ff-only'],
                           capture_output=True, text=True, env=_SETUP_ENV)
        print('      ' + (r.stdout.strip() or 'Already up to date'))
    else:
        r = subprocess.run(
            ['git', 'clone', '--depth', '1', '-b', 'main',
             'https://github.com/rajatsarswat2001/autopilot.git', CLONE_DIR],
            capture_output=True, text=True, env=_SETUP_ENV
        )
        if r.returncode == 0:
            print('      ✅ Cloned main OK')
        else:
            print('      ❌ Clone failed: ' + r.stderr[-200:])
            raise RuntimeError("Git clone failed!")

    # ── 2. Run kaggle_setup.py (passes eager env to all pip subprocesses) ────
    print("\n[2/3] Running kaggle_setup.py...")
    setup_path = os.path.join(CLONE_DIR, 'kaggle_setup.py')
    subprocess.run([sys.executable, setup_path],
                   capture_output=False, env=_SETUP_ENV)

    # ── 3. Kernel restart — required to load newly installed binaries ────────
    print("\n[3/3] Restarting kernel to activate new packages...")
    print("🔄 KERNEL RESTARTING — wait ~3 seconds, then run Cell 2 directly.")
    print("=" * 60)
    os._exit(0)


[0/6] Pinning numpy==1.26.4 (prevents numpy 2.x binary mismatch)...
  OK  numpy pinned to 1.26.4
🔧 Environment needs setup. Running installer...

[1/3] Getting latest code from branch 'main'...
      ✅ Cloned main OK

[2/3] Running kaggle_setup.py...


2026-06-02 13:28:38.763257: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780406918.958680     135 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780406919.018207     135 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780406919.459474     135 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780406919.459523     135 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780406919.459526     135 computation_placer.cc:177] computation placer alr


AutoPilot -- Kaggle T4 Environment Setup

[0/7] Pre-pinning numpy==1.26.4 ...
  [OK  ]  numpy==1.26.4 (pre-pin)  (6s)

[1/7] System packages (ffmpeg, libsndfile, espeak-ng) ...
  [OK ]  ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers

[2/7] Core pipeline packages (independent install groups) ...
  [OK  ]  utilities  (4s)
  [OK  ]  API clients  (4s)
  [OK  ]  edge-tts  (3s)
  [OK  ]  langchain-core 1.x  (3s)
  [OK  ]  langchain 1.x  (3s)
  [OK  ]  langchain-community  (9s)
  [OK  ]  langgraph 1.x  (3s)

[3/7] GPU packages (upgrading diffusers 0.29 -> 0.33) ...
  [OK  ]  diffusers==0.33.0 --no-deps  (4s)
  [OK  ]  diffusers deps  (17s)
  [OK  ]  bitsandbytes (optional CogVideoX fallback)  (5s)

[4/7] Chatterbox TTS (--no-deps + manual dep pinning) ...
  [OK  ]  chatterbox-tts --no-deps  (2s)
  [OK  ]  conformer==0.3.2  (3s)
  [OK  ]  resemble-perth==1.0.1  (4s)
  [OK  ]  librosa  (3s)
  [OK  ]  s3tokenizer --no-deps  (1s)
  [OK  ]  onnx==1.16.0  (14s)

In [1]:
import subprocess
print("Pulling latest Kaggle fixes...")
r = subprocess.run(
    ["git", "-C", "/kaggle/working/autopilot", "pull", "origin", "main"], 
    capture_output=True, text=True
)
print(r.stdout)
print(r.stderr)

Pulling latest Kaggle fixes...
Already up to date.

From https://github.com/rajatsarswat2001/autopilot
 * branch            main       -> FETCH_HEAD



In [ ]:
# =============================================================================
# CELL 2 -- CLONE COMFYUI & DOWNLOAD WAN 2.2 TI2V 5B MODELS
# =============================================================================
# Downloads 4 files via aria2c (16 parallel connections, resumable):
#   wan2.2_ti2v_5B_fp16.safetensors  (~9.3 GB)  -- the AI video brain
#   umt5_xxl_fp8_e4m3fn_scaled       (~6.3 GB)  -- text encoder
#   wan2.2_vae.safetensors           (~1.3 GB)  -- video decoder
#   clip_vision_h.safetensors        (~0.6 GB)  -- image conditioning
# =============================================================================
import os, sys, subprocess, time, torch

# -- Install aria2 for fast downloads --
os.system('apt-get update && apt-get install -y aria2')

# -- GPU info --
if torch.cuda.is_available():
    n = torch.cuda.device_count()
    for i in range(n):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name}  {p.total_memory/1024**3:.1f} GB')
    print()

# -- [1/3] Clone ComfyUI --
COMFY_DIR = '/kaggle/working/ComfyUI'
if not os.path.exists(COMFY_DIR):
    print('[1/3] Cloning ComfyUI...')
    subprocess.run(['git', 'clone', 'https://github.com/comfyanonymous/ComfyUI.git', COMFY_DIR],
                   capture_output=True)
    print('  OK  ComfyUI cloned')
    print('  Installing ComfyUI requirements...')
    subprocess.run(['pip', 'install', '-r', f'{COMFY_DIR}/requirements.txt'], capture_output=True)
else:
    print('[1/3] ComfyUI already present')
sys.path.insert(0, COMFY_DIR)

# -- [2/3] Create model dirs --
DIRS = {
    'diffusion': f'{COMFY_DIR}/models/diffusion_models',
    'text_enc' : f'{COMFY_DIR}/models/text_encoders',
    'vae'      : f'{COMFY_DIR}/models/vae',
    'clip_vis' : f'{COMFY_DIR}/models/clip_vision',
}
for d in DIRS.values(): os.makedirs(d, exist_ok=True)

# -- [3/3] Download Wan 2.2 weights --
print('[3/3] Downloading Wan 2.2 model weights...')
BASE     = 'https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files'
BASE21   = 'https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files'

DOWNLOADS = [
    (f'{BASE}/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors',        DIRS['diffusion'], 'wan2.2_ti2v_5B_fp16.safetensors'),
    (f'{BASE}/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors',   DIRS['text_enc'],  'umt5_xxl_fp8_e4m3fn_scaled.safetensors'),
    (f'{BASE}/vae/wan2.2_vae.safetensors',                              DIRS['vae'],       'wan2.2_vae.safetensors'),
    (f'{BASE21}/clip_vision/clip_vision_h.safetensors',                 DIRS['clip_vis'],  'clip_vision_h.safetensors'),
]

for url, dest_dir, fname in DOWNLOADS:
    dest = os.path.join(dest_dir, fname)
    if os.path.exists(dest):
        size_gb = os.path.getsize(dest) / 1024**3
        print(f'  SKIP  {fname} ({size_gb:.1f} GB cached)')
    else:
        print(f'  DL    {fname} ...')
        t0 = time.time()
        os.system(f"aria2c --console-log-level=error -c -x 16 -s 16 -k 1M '{url}' -d '{dest_dir}' -o '{fname}'")
        elapsed = time.time() - t0
        size_gb = os.path.getsize(dest) / 1024**3 if os.path.exists(dest) else 0
        print(f'  OK    {fname} ({size_gb:.1f} GB in {elapsed:.0f}s)')

print()
print('=== Wan 2.2 models ready ===')
print('VRAM allocated (should be 0 — no model loaded yet):', torch.cuda.memory_allocated(0) / 1e9, 'GB')
print('Proceed to Cell 3 to configure API keys.')


In [ ]:
# =============================================================================
# CELL 3 -- API KEYS & SETTINGS  (Dual-GPU Wan 2.2 + 7-Tier LLM Fallback)
# =============================================================================
# LLM Waterfall (auto key rotation within each provider):
#   1. Groq        (GROQ_API_KEYS     -- multiple keys, rotate on 429)
#   2. Gemini      (GEMINI_API_KEYS   -- multiple keys, rotate on 429)
#   3. DeepSeek    (DEEPSEEK_API_KEY)
#   4. NVIDIA NIM  (NVIDIA_API_KEY)
#   5. OpenAI      (OPENAI_API_KEY    -- if available)
#   6. Ollama      (LOCAL -- llama3:8b-instruct-q4_K_M, always available)
# =============================================================================
import os, subprocess, sys, time, torch

PIPELINE_DIR = '/kaggle/working/autopilot/autopilot_pipeline'

# -- Exact Kaggle Secret names --
KAGGLE_SECRET_KEYS = [
    'DEEPSEEK_API_KEY',
    'GEMINI_API_KEYS',
    'GROQ_API_KEYS',
    'HF_TOKEN',
    'NVIDIA_API_KEY',
    'PEXELS_API_KEY',
    'PIXABAY_KEY',
    'TAVILY_API_KEY',
    'YOUTUBE_API_KEY',
]

env_values = {}

print('Loading secrets from Kaggle Secrets...')
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    for k in KAGGLE_SECRET_KEYS:
        try:
            val = user_secrets.get_secret(k)
            if val:
                env_values[k] = val
                masked = val[:6] + '***' if len(val) > 6 else '***'
                print(f'  OK  {k:<25} {masked}')
        except Exception:
            print(f'  --  {k:<25} (not set)')
except ImportError:
    print('  Not in Kaggle -- skipping secrets')

# Pipeline uses plural for Pexels (read from singular Kaggle Secret)
if 'PEXELS_API_KEY' in env_values:
    env_values['PEXELS_API_KEYS'] = env_values['PEXELS_API_KEY']
# Pipeline uses PIXABAY_API_KEY -- map from PIXABAY_KEY
if 'PIXABAY_KEY' in env_values:
    env_values['PIXABAY_API_KEY'] = env_values['PIXABAY_KEY']
# HuggingFace token for model downloads
if 'HF_TOKEN' in env_values:
    os.environ['HF_TOKEN'] = env_values['HF_TOKEN']
    os.environ['HUGGING_FACE_HUB_TOKEN'] = env_values['HF_TOKEN']

# ==========================================================================
# Install & start Ollama (local LLM fallback -- no API key, no rate limits)
# ==========================================================================
print('\nSetting up Ollama (local LLM fallback)...')
try:
    # Install ollama binary if not present
    if not os.path.exists('/usr/local/bin/ollama'):
        subprocess.run(
            'curl -fsSL https://ollama.com/install.sh | sh',
            shell=True, capture_output=True
        )
    # Start ollama server in background
    subprocess.Popen(['ollama', 'serve'],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(3)
    # Pull the model (cached after first run)
    print('  Pulling mistral-nemo:12b (cached after first run, ~7.5 GB)...')
    r = subprocess.run(
        ['ollama', 'pull', 'mistral-nemo:12b'],
        capture_output=True, text=True, timeout=600
    )
    if r.returncode == 0:
        print('  OK  Ollama mistral-nemo:12b ready (local fallback active)')
    else:
        print('  WARN ollama pull failed:', r.stderr[:200])
except Exception as e:
    print(f'  WARN Ollama setup failed (non-fatal): {e}')

# ==========================================================================
# Pipeline settings
# ==========================================================================
has_gpu = torch.cuda.is_available()
n_gpus  = torch.cuda.device_count() if has_gpu else 0

pipeline_settings = {
    # Core
    'AUTOPILOT_AUTO_APPROVE':  '1',
    'FFMPEG_PRESET':           'ultrafast',
    'LOG_LEVEL':               'INFO',
    'FORMAT':                  'short',
    # Video generation: Wan 2.2 TI2V 5B via ComfyUI dual-GPU
    'VIDEO_GEN_ENABLED':       '1' if has_gpu else '0',
    'VIDEO_GEN_MODEL':         'wan22',
    'VIDEO_GEN_WAN_STEPS':     '20',
    'MAX_VIDEO_DURATION_S':    '20',
    'SCRIPT_SCENE_COUNT':      '3',
    'KAGGLE_NGROK_URL':        'http://127.0.0.1:8080',
    # Parallelism
    'AUDIO_PARALLEL_WORKERS':  '1',
    'VISUAL_PARALLEL_WORKERS': '2' if n_gpus >= 2 else '1',
    # Stock footage
    'DISABLE_STOCK':           '0',
}
env_values.update(pipeline_settings)

# Write .env + export
os.makedirs(PIPELINE_DIR, exist_ok=True)
env_path = os.path.join(PIPELINE_DIR, '.env')
with open(env_path, 'w') as f:
    for k, v in env_values.items():
        f.write(f'{k}={v}\n')
        os.environ[k] = str(v)

print()
print('=' * 60)
print(f'  .env          -> {env_path}')
print(f'  GPUs          -> {n_gpus}x T4')
print(f'  Video model   -> {pipeline_settings["VIDEO_GEN_MODEL"]} @ {pipeline_settings["KAGGLE_NGROK_URL"]}')
print(f'  LLM workers   -> VISUAL={pipeline_settings["VISUAL_PARALLEL_WORKERS"]}')
print(f'  LLM fallback  -> Groq -> Gemini -> DeepSeek -> NVIDIA -> Ollama (local)')
print('=' * 60)
print('Run Cell 4 (cleanup) -> Cell 5 (GPU workers) -> Cell 6 (pipeline)')


In [4]:
# =============================================================================
# CLEANUP CACHE BEFORE RUNNING PIPELINE
# =============================================================================
import shutil, os, subprocess

# Clear model cache of unused models (old Wan 2.1 diffusers + any leftovers)
cache_dirs_to_clean = [
    "/root/.cache/huggingface/hub/models--Wan-AI--Wan2.1-T2V-1.3B-Diffusers",
    "/root/.cache/huggingface/hub/models--Lightricks--LTX-Video",
]
for d in cache_dirs_to_clean:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"Deleted: {d}")

# Clear scratch outputs
scratch = "/kaggle/working/autopilot/autopilot_pipeline/outputs/video/scratch"
if os.path.exists(scratch):
    shutil.rmtree(scratch)
    os.makedirs(scratch)
    print("Scratch cleared")

# Check disk
result = subprocess.run(["df", "-h", "/kaggle/working"], capture_output=True, text=True)
print(result.stdout)


Deleted: /root/.cache/huggingface/hub/models--Wan-AI--Wan2.1-T2V-1.3B-Diffusers
Scratch cleared
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  1.5M   20G   1% /kaggle/working



In [ ]:
# =============================================================================
# CELL 5 -- LAUNCH DUAL-GPU WAN 2.2 COMFYUI WORKERS  (run before pipeline)
# =============================================================================
# Spawns:
#   worker.py 8001  (CUDA_VISIBLE_DEVICES=0) -- GPU 0
#   worker.py 8002  (CUDA_VISIBLE_DEVICES=1) -- GPU 1
#   balancer.py 8080                         -- round-robin load balancer
#
# AutoPilot pipeline sends POST /generate_video to localhost:8080
# The balancer fans requests out to GPU 0 and GPU 1 in parallel.
# 2 clips render simultaneously -> 2x speed!
# =============================================================================
import os, sys, time, subprocess, requests

WORKER_PATH   = '/kaggle/working/wan22_worker.py'
BALANCER_PATH = '/kaggle/working/wan22_balancer.py'

# ── Write worker.py ──────────────────────────────────────────────────────────
worker_code = '''
import gc, os, random, sys, torch, imageio, numpy as np
from pathlib import Path
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import FileResponse
import uvicorn

sys.path.insert(0, "/kaggle/working/ComfyUI")
from nodes import NODE_CLASS_MAPPINGS

OUTPUT_DIR = "/kaggle/working/wan22_output"
INPUT_DIR  = "/kaggle/working/wan22_input"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(INPUT_DIR,  exist_ok=True)

app = FastAPI(title="Wan2.2 Worker")

@app.get("/health")
def health(): return {"status": "ok", "device": os.environ.get("CUDA_VISIBLE_DEVICES", "0")}

@app.post("/generate_video")
async def gen(
    image:  UploadFile = File(None),
    prompt: str = Form(...),
    seed:   int = Form(0),
    steps:  int = Form(30),
    width:  int = Form(832),
    height: int = Form(480),
    frames: int = Form(49),
):
    if seed == 0: seed = random.randint(0, 2**32 - 1)
    with torch.inference_mode():
        clip = NODE_CLASS_MAPPINGS["CLIPLoader"]().load_clip(
            "umt5_xxl_fp8_e4m3fn_scaled.safetensors", "wan", "default")[0]
        pos_c = NODE_CLASS_MAPPINGS["CLIPTextEncode"]().encode(clip, prompt)[0]
        neg_c = NODE_CLASS_MAPPINGS["CLIPTextEncode"]().encode(clip, "text, watermark, blurry")[0]
        del clip; gc.collect(); torch.cuda.empty_cache()

        loaded_image, clip_vis_out = None, None
        if image and image.filename:
            img_path = f"{INPUT_DIR}/{image.filename}"
            with open(img_path, "wb") as fh: fh.write(await image.read())
            loaded_image = NODE_CLASS_MAPPINGS["LoadImage"]().load_image(img_path)[0]
            cv = NODE_CLASS_MAPPINGS["CLIPVisionLoader"]().load_clip("clip_vision_h.safetensors")[0]
            clip_vis_out = NODE_CLASS_MAPPINGS["CLIPVisionEncode"]().encode(cv, loaded_image, "none")[0]
            del cv; gc.collect(); torch.cuda.empty_cache()

        vae = NODE_CLASS_MAPPINGS["VAELoader"]().load_vae("wan2.2_vae.safetensors")[0]
        wan_cls = NODE_CLASS_MAPPINGS.get("WanImageToVideo")
        if loaded_image is not None and wan_cls:
            pos_c, neg_c, lat = wan_cls().encode(pos_c, neg_c, vae, width, height, frames, 1, loaded_image, clip_vis_out)
        else:
            lat = NODE_CLASS_MAPPINGS["EmptyLatentImage"]().generate(width, height, 1)[0]

        model = NODE_CLASS_MAPPINGS["UNETLoader"]().load_unet("wan2.2_ti2v_5B_fp16.safetensors", "default")[0]
        sampled = NODE_CLASS_MAPPINGS["KSampler"]().sample(
            model, seed, steps, 6.0, "euler", "simple", pos_c, neg_c, lat, 1.0)[0]
        del model; gc.collect(); torch.cuda.empty_cache()

        decoded = NODE_CLASS_MAPPINGS["VAEDecode"]().decode(vae, sampled)[0]
        del vae, sampled; gc.collect(); torch.cuda.empty_cache()

        out = f"{OUTPUT_DIR}/wan22_{seed}.mp4"
        with imageio.get_writer(out, fps=16) as w:
            for fr in decoded: w.append_data((fr.cpu().numpy()*255).astype(np.uint8))
        return FileResponse(out, media_type="video/mp4", filename="output.mp4")

if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=int(sys.argv[1]))
'''
with open(WORKER_PATH, 'w') as f: f.write(worker_code)

# ── Write balancer.py ────────────────────────────────────────────────────────
balancer_code = '''
import asyncio, httpx, uvicorn
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import FileResponse

app     = FastAPI(title="Wan2.2 Load Balancer")
WORKERS = ["http://127.0.0.1:8001", "http://127.0.0.1:8002"]
_idx    = 0
_lock   = asyncio.Lock()

@app.get("/health")
def health(): return {"status": "ok", "workers": WORKERS}

@app.post("/generate_video")
async def gen(
    image: UploadFile = File(None), prompt: str = Form(...),
    seed: int = Form(0), steps: int = Form(30),
    width: int = Form(832), height: int = Form(480), frames: int = Form(49)
):
    global _idx
    async with _lock:
        target = WORKERS[_idx % len(WORKERS)]
        _idx  += 1
    data  = {"prompt": prompt, "seed": str(seed), "steps": str(steps),
             "width": str(width), "height": str(height), "frames": str(frames)}
    files = {}
    if image and image.filename:
        files = {"image": (image.filename, await image.read(), image.content_type)}
    async with httpx.AsyncClient(timeout=3600.0) as c:
        r = await c.post(f"{target}/generate_video", data=data, files=files)
    out = f"/kaggle/working/wan22_output/bal_{seed}.mp4"
    with open(out, "wb") as f: f.write(r.content)
    return FileResponse(out, media_type="video/mp4")

if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=8080)
'''
with open(BALANCER_PATH, 'w') as f: f.write(balancer_code)

# ── Launch workers ───────────────────────────────────────────────────────────
import torch
n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0

env0 = {**os.environ, 'CUDA_VISIBLE_DEVICES': '0'}
env1 = {**os.environ, 'CUDA_VISIBLE_DEVICES': '1'}

print('Launching Wan 2.2 ComfyUI workers...')
p_bal = subprocess.Popen([sys.executable, BALANCER_PATH])
p_w0  = subprocess.Popen([sys.executable, WORKER_PATH, '8001'], env=env0)
p_w1  = subprocess.Popen([sys.executable, WORKER_PATH, '8002'], env=env1) if n_gpus >= 2 else None

# ── Wait for all workers to be healthy ───────────────────────────────────────
ports = [8080, 8001] + ([8002] if n_gpus >= 2 else [])
for port in ports:
    print(f'  Waiting for port {port}...', end=' ', flush=True)
    for _ in range(60):
        try:
            r = requests.get(f'http://127.0.0.1:{port}/health', timeout=2)
            if r.status_code == 200:
                print('UP')
                break
        except:
            time.sleep(3)
    else:
        print('TIMEOUT -- check worker logs')

print()
print('=' * 60)
print('DUAL-GPU WORKERS READY')
print(f'  GPU 0  -> http://127.0.0.1:8001')
if n_gpus >= 2:
    print(f'  GPU 1  -> http://127.0.0.1:8002')
print(f'  Balancer -> http://127.0.0.1:8080')
print('=' * 60)
print('\nNow run Cell 6 to start the AutoPilot pipeline!')


In [ ]:
# =============================================================================
# TESTING MODE — VIDEO DURATION CAP + PRE-WRITTEN SCRIPT OPTION
# =============================================================================
# Run this cell AFTER Cell 3 (keys) and BEFORE Cell 4 (run pipeline).
# It overrides env vars to cap total video to 15-20 seconds.
# =============================================================================
import os, subprocess, sys

PIPELINE_DIR = '/kaggle/working/autopilot/autopilot_pipeline'

# ─── Pull latest code first ─────────────────────────────────────────────────
print("Pulling latest code from GitHub...")
r = subprocess.run(
    ["git", "-C", "/kaggle/working/autopilot", "pull", "origin", "main"],
    capture_output=True, text=True
)
print(r.stdout.strip() or "Already up to date.")
if r.returncode != 0:
    print("WARN:", r.stderr[:200])

# ─── Testing phase overrides ─────────────────────────────────────────────────
testing_overrides = {
    # === DURATION CAP (15-20 sec total) ===
    "MAX_VIDEO_DURATION_S":  "20",   # hard cap: each clip ≤20s, total ≤20s
    "SCRIPT_SCENE_COUNT":    "3",    # 3 scenes × ~5-7s narration each ≈ 15-20s

    # === WAN2.1 GENERATION SETTINGS ===
    "VIDEO_GEN_ENABLED":     "1",
    "VIDEO_GEN_MODEL":       "wan21",
    "VIDEO_GEN_WAN_STEPS":   "20",   # reduced from 30 for faster testing
    "VIDEO_GEN_WAN_GUIDANCE":"5.0",

    # === PERFORMANCE ===
    "AUDIO_PARALLEL_WORKERS":"1",
    "VISUAL_PARALLEL_WORKERS":"1",
    "FFMPEG_PRESET":          "ultrafast",

    # === STOCK FOOTAGE (fast fallback if GPU fails) ===
    "DISABLE_STOCK":          "0",

    # === MISC ===
    "AUTOPILOT_AUTO_APPROVE": "1",
    "LOG_LEVEL":              "INFO",
    "FORMAT":                 "short",
}

# Apply all overrides
for k, v in testing_overrides.items():
    os.environ[k] = v

# Rewrite .env so pipeline subprocess picks them up
env_path = os.path.join(PIPELINE_DIR, '.env')
os.makedirs(PIPELINE_DIR, exist_ok=True)
existing = {}
if os.path.exists(env_path):
    with open(env_path) as f:
        for line in f:
            line = line.strip()
            if line and '=' in line and not line.startswith('#'):
                k, _, v = line.partition('=')
                existing[k.strip()] = v.strip()
existing.update(testing_overrides)
with open(env_path, 'w') as f:
    f.write("# AutoPilot .env — TESTING PHASE (auto-generated)\n")
    for k, v in existing.items():
        f.write(f"{k}={v}\n")

print()
print("=" * 60)
print("TESTING MODE ACTIVE")
print(f"  Max video duration : {testing_overrides['MAX_VIDEO_DURATION_S']}s")
print(f"  Scenes per video   : {testing_overrides['SCRIPT_SCENE_COUNT']}")
print(f"  Wan2.1 steps       : {testing_overrides['VIDEO_GEN_WAN_STEPS']}")
print(f"  FFMPEG preset      : {testing_overrides['FFMPEG_PRESET']}")
print(f"  Stock footage      : {'disabled' if testing_overrides['DISABLE_STOCK']=='1' else 'enabled (fallback)'}")
print("=" * 60)
print()
print("Clip duration math:")
fps = 24
max_s = int(testing_overrides["MAX_VIDEO_DURATION_S"])
frames = max_s * fps
# snap to odd
frames = frames if frames % 2 == 1 else frames - 1
print(f"  {max_s}s × {fps}fps = {frames} frames (odd-snapped for Wan2.1)")
print()
print("Estimated pipeline time on T4:")
scenes = int(testing_overrides["SCRIPT_SCENE_COUNT"])
print(f"  {scenes} scenes × 2 clips × ~3-5 min/clip = ~{scenes*2*4} min total")
print()
print("Now run Cell 4 (pipeline) →")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 4 — RUN PIPELINE
#
# GPU mode (CogVideoX-2B):  ~35-45 min per 60s video on T4
# CPU mode (Pexels only): ~8-12 min per 60s video
# ═══════════════════════════════════════════════════════════════════════════
import subprocess, sys, os, time
from pathlib import Path

PIPELINE_DIR = '/kaggle/working/autopilot/autopilot_pipeline'

# ── CONFIG ───────────────────────────────────────────────────────────────────
# personal_finance | saas_tools | legal_tax | senior_health | storytelling
NICHE = 'personal_finance'
TOPIC = ''     # leave empty = auto-detect trending topic via Tavily + pytrends

# ── GPU VRAM check ───────────────────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    used = torch.cuda.memory_allocated(0) / 1e9
    free = vram - used
    print(f'VRAM: {free:.1f} GB free / {vram:.1f} GB total')
    if free < 8:
        print('⚠️  Low VRAM — forcing VIDEO_GEN_ENABLED=0 (using Pexels instead)')
        os.environ['VIDEO_GEN_ENABLED'] = '0'
    else:
        print('✅ Sufficient VRAM for CogVideoX-2B + Chatterbox')
else:
    print('⚠️  No GPU — running in CPU-only mode (Pexels + Edge TTS)')

# ── Pre-run checks ───────────────────────────────────────────────────────────
checks = {
    'Pipeline dir': os.path.exists(PIPELINE_DIR),
    'main.py':      os.path.exists(os.path.join(PIPELINE_DIR, 'main.py')),
    '.env':         os.path.exists(os.path.join(PIPELINE_DIR, '.env')),
}
print('\nPRE-RUN CHECKS')
for k, v in checks.items():
    print(f'  {"✅" if v else "❌"} {k}')
    if not v:
        raise RuntimeError(f'{k} missing — run Cell 1 and Cell 3 first')

print(f'  Niche: {NICHE}')
print(f'  Topic: {TOPIC or "auto-detect"}')
print('=' * 60)

# Set NICHE as env var so visual_director picks it up
os.environ['NICHE'] = NICHE

cmd = [
    sys.executable, 'main.py',
    '--niche', NICHE,
    '--no-db',
    '--approve',
    '--log-format', 'console',
]
if TOPIC:
    cmd += ['--topic', TOPIC]

start = time.time()
proc = subprocess.Popen(
    cmd,
    cwd=PIPELINE_DIR,
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1
)

all_lines = []
for line in proc.stdout:
    print(line, end='', flush=True)
    all_lines.append(line)
proc.wait()

elapsed = time.time() - start
print('=' * 60)
print(f'EXIT CODE : {proc.returncode}')
print(f'DURATION  : {elapsed:.0f}s ({elapsed/60:.1f} min)')

if proc.returncode != 0:
    print('\n❌ FAILED — last 30 lines:')
    print(''.join(all_lines[-30:]))
else:
    print('\n✅ SUCCESS!')
    vid_dir = Path(PIPELINE_DIR) / 'outputs' / 'video'
    videos = sorted(vid_dir.glob('*.mp4'), key=lambda x: x.stat().st_mtime, reverse=True)
    print(f'Videos generated: {len(videos)}')
    for v in videos[:5]:
        mb = v.stat().st_size / 1024 / 1024
        thumb = vid_dir / (v.stem + '_thumb.jpg')
        has_thumb = '🖼️' if thumb.exists() else ''
        print(f'  {v.name}  ({mb:.1f} MB) {has_thumb}')
    print('\nRun Cell 5 to view thumbnails and download')

VRAM: 15.6 GB free / 15.6 GB total
✅ Sufficient VRAM for CogVideoX-2B + Chatterbox

PRE-RUN CHECKS
  ✅ Pipeline dir
  ✅ main.py
  ✅ .env
  Niche: personal_finance
  Topic: auto-detect
/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
INFO:main:13:33:01 [info     ] main.checkpointer              type=memory
INFO:main:13:33:01 [info     ] main.start                     niche=personal_finance thread_id=46912e89-ca5a-4eca-8dad-d9baa466ceed topic=None
INFO:agents.trend_scout:13:33:01 [info     ] trend_scout.start              default_tier=1 niche=personal_finance
INFO:agents.trend_scout:13:33:05 [info     ] trend_scout.selected           score=4.6 tier=1 topic="5 Easily Fixable Finan

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 5 — VIEW & DOWNLOAD RESULTS
# ═══════════════════════════════════════════════════════════════════════════
from pathlib import Path
from IPython.display import HTML, Image, display
import base64

PIPELINE_DIR = '/kaggle/working/autopilot/autopilot_pipeline'
vid_dir  = Path(PIPELINE_DIR) / 'outputs' / 'video'
videos   = sorted(vid_dir.glob('*.mp4'), key=lambda x: x.stat().st_mtime, reverse=True)

if not videos:
    print('No videos yet — run Cell 4 first')
else:
    total_mb = sum(v.stat().st_size for v in videos) / 1024 / 1024
    print(f'Found {len(videos)} videos ({total_mb:.1f} MB total)')
    print('Download from the Output tab (right panel in Kaggle UI)\n')

    rows = []
    for v in videos:
        mb   = v.stat().st_size / 1024 / 1024
        thumb = vid_dir / (v.stem + '_thumb.jpg')
        ass   = vid_dir / (v.stem + '_captions.ass')

        if thumb.exists():
            img_data = base64.b64encode(thumb.read_bytes()).decode()
            thumb_html = f'<img src="data:image/jpeg;base64,{img_data}" width="240" style="border-radius:8px">'
        else:
            thumb_html = '<div style="width:240px;height:135px;background:#333;border-radius:8px;display:flex;align-items:center;justify-content:center;color:#888">No thumbnail</div>'

        caption_badge = '✅ Captions' if ass.exists() else ''
        rows.append(
            f'<tr style="border-bottom:1px solid #333">'
            f'<td style="padding:10px">{thumb_html}</td>'
            f'<td style="padding:10px;vertical-align:top">'
            f'<b style="font-size:14px">{v.name}</b><br>'
            f'<span style="color:#aaa">{mb:.1f} MB</span><br>'
            f'<span style="color:#4CAF50">{caption_badge}</span>'
            f'</td></tr>'
        )

    html = (
        '<div style="background:#1a1a1a;padding:20px;border-radius:12px">'
        '<h2 style="color:#fff;margin-top:0">🎬 Generated Videos</h2>'
        '<table style="border-collapse:collapse;width:100%">'
        + ''.join(rows)
        + '</table></div>'
    )
    display(HTML(html))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 6 — BATCH MODE  (multiple niches, run overnight)
#
# T4 GPU (30 hrs/week quota) can produce ~40-45 videos/week
# GPU mode: ~40 min/video → 6 hr = ~9 videos overnight
# CPU mode: ~10 min/video → 6 hr = ~36 videos overnight (Pexels only)
# ═══════════════════════════════════════════════════════════════════════════
import subprocess, sys, os, time
from pathlib import Path

PIPELINE_DIR = '/kaggle/working/autopilot/autopilot_pipeline'

# ── CONFIG ───────────────────────────────────────────────────────────────────
# Each entry: (niche, topic_override_or_None)
BATCH_JOBS = [
    ('personal_finance',  None),   # auto-detect trending topic
    ('saas_tools',        None),
    ('personal_finance',  None),   # second video, different trending topic
    # ('legal_tax',       None),
    # ('senior_health',   None),
    # ('storytelling',    'The untold story of the Manhattan Project'),
]

import torch
gpu_mode = torch.cuda.is_available()
est_min  = 40 if gpu_mode else 10
print(f'Batch: {len(BATCH_JOBS)} videos')
print(f'Mode: {"GPU (Wan2.1 + Chatterbox)" if gpu_mode else "CPU (Pexels + Edge TTS)"}')
print(f'Estimated time: ~{len(BATCH_JOBS) * est_min} min ({len(BATCH_JOBS) * est_min / 60:.1f} hrs)')
print('=' * 60)

results = []
total_start = time.time()

for i, (niche, topic) in enumerate(BATCH_JOBS, 1):
    print(f'\n[{i}/{len(BATCH_JOBS)}] niche={niche} topic={topic or "auto"} ...')
    os.environ['NICHE'] = niche
    t0 = time.time()

    cmd = [
        sys.executable, 'main.py',
        '--niche', niche,
        '--no-db', '--approve',
        '--log-format', 'console'
    ]
    if topic:
        cmd += ['--topic', topic]

    r = subprocess.run(
        cmd,
        cwd=PIPELINE_DIR,
        env=os.environ.copy(),
        timeout=7200,  # 2hr max per video
        capture_output=True, text=True
    )
    elapsed = time.time() - t0
    ok = r.returncode == 0
    results.append((niche, topic or 'auto', ok, elapsed))
    status = '✅' if ok else '❌'
    print(f'  {status} ({elapsed:.0f}s / {elapsed/60:.1f} min)')
    if not ok:
        lines = (r.stdout + r.stderr).split('\n')
        print('\n'.join(lines[-10:]))

    # Clear GPU between videos
    if gpu_mode:
        import gc
        torch.cuda.empty_cache()
        gc.collect()

total_elapsed = time.time() - total_start
print(f'\n{"="*60}')
print(f'BATCH COMPLETE  ({total_elapsed/60:.1f} min total)')
success = sum(1 for _, _, ok, _ in results if ok)
print(f'Results: {success}/{len(results)} succeeded')
for niche, topic, ok, t in results:
    print(f'  {"✅" if ok else "❌"}  {niche:<25}  {topic:<20}  {t/60:.1f} min')

vid_dir = Path(PIPELINE_DIR) / 'outputs' / 'video'
videos  = sorted(vid_dir.glob('*.mp4'), key=lambda x: x.stat().st_mtime, reverse=True)
print(f'\nTotal videos in output: {len(videos)}')
for v in videos:
    print(f'  {v.name}  ({v.stat().st_size/1024/1024:.1f} MB)')